In [ ]:
# 1. 라이브러리
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
# 2. 경로 설정
# 프로젝트 최상위 폴더 탐색
current_path = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        path
        for path in [current_path, *current_path.parents]
        if (path / "data" / "raw").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "data/raw 폴더를 찾을 수 없습니다."
    )

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
REPORT_TABLE_DIR = PROJECT_ROOT / "reports" / "tables"
REPORT_FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
REPORT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
REPORT_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

BUSINESS_JSON = (
    RAW_DIR
    / "yelp_academic_dataset_business.json"
)

print("Business 파일:", BUSINESS_JSON)
print("파일 존재:", BUSINESS_JSON.exists())

Business 파일: C:\Users\playdata2\SKN34-2nd-5Team\data\raw\yelp_academic_dataset_business.json
파일 존재: True


In [4]:
# 3. Business 데이터 불러오기
business_df = pd.read_json(
    BUSINESS_JSON,
    lines=True
)

print("Business 데이터 크기:", business_df.shape)

business_df.head()

Business 데이터 크기: (150346, 14)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,{'ByAppointmentOnly': 'True'},"Doctors, Traditional Chinese Medicine, Naturop...",None
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,{'BusinessAcceptsCreditCards': 'True'},"Shipping Centers, Local Services, Notaries, Ma...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-18:30', ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{'BikeParking': 'True', 'BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{'Monday': '8:0-22:0', 'Tuesday': '8:0-22:0', ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{'RestaurantsDelivery': 'False', 'OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{'Monday': '7:0-20:0', 'Tuesday': '7:0-20:0', ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{'BusinessAcceptsCreditCards': 'True', 'Wheelc...","Brewpubs, Breweries, Food","{'Wednesday': '14:0-22:0', 'Thursday': '16:0-2..."


In [5]:
# 4. 컬럼과 결측 확인
business_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150346 entries, 0 to 150345
Data columns (total 14 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   business_id   150346 non-null  str    
 1   name          150346 non-null  str    
 2   address       150346 non-null  str    
 3   city          150346 non-null  str    
 4   state         150346 non-null  str    
 5   postal_code   150346 non-null  str    
 6   latitude      150346 non-null  float64
 7   longitude     150346 non-null  float64
 8   stars         150346 non-null  float64
 9   review_count  150346 non-null  int64  
 10  is_open       150346 non-null  int64  
 11  attributes    136602 non-null  object 
 12  categories    150243 non-null  str    
 13  hours         127123 non-null  object 
dtypes: float64(3), int64(2), object(2), str(7)
memory usage: 35.4+ MB


In [6]:
business_missing_df = (
    business_df
    .isna()
    .sum()
    .to_frame(name="missing_count")
)

business_missing_df["missing_rate_pct"] = (
    business_missing_df["missing_count"]
    / len(business_df)
    * 100
).round(2)

business_missing_df.sort_values(
    "missing_rate_pct",
    ascending=False
)

,missing_count,missing_rate_pct
hours,23223,15.45
attributes,13744,9.14
categories,103,0.07
address,0,0.00
business_id,0,0.00
name,0,0.00
postal_code,0,0.00
state,0,0.00
city,0,0.00
latitude,0,0.00


In [8]:
business_df[
    [
        "business_id",
        "name",
        "city",
        "state",
        "latitude",
        "longitude",
        "stars",
        "review_count",
        "is_open",
        "categories"
    ]
].head()

,business_id,name,city,state,latitude,longitude,stars,review_count,is_open,categories
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ",Santa Barbara,CA,34.426679,-119.711197,5.0,7,0,"Doctors, Traditional Chinese Medicine, Naturop..."
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,Affton,MO,38.551126,-90.335695,3.0,15,1,"Shipping Centers, Local Services, Notaries, Ma..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,Tucson,AZ,32.223236,-110.880452,3.5,22,0,"Department Stores, Shopping, Fashion, Home & G..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,Philadelphia,PA,39.955505,-75.155564,4.0,80,1,"Restaurants, Food, Bubble Tea, Coffee & Tea, B..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,Green Lane,PA,40.338183,-75.471659,4.5,13,1,"Brewpubs, Breweries, Food"


In [9]:
# 5. 카테고리 결측 확인
category_null_count = (
    business_df["categories"].isna().sum()
)

category_null_rate = (
    category_null_count
    / len(business_df)
    * 100
)

print("카테고리 결측 수:", category_null_count)
print(
    f"카테고리 결측률: {category_null_rate:.4f}%"
)

카테고리 결측 수: 103
카테고리 결측률: 0.0685%


In [10]:
# 6. 카테고리를 리스트로 변환
business_df["category_list"] = (
    business_df["categories"]
    .fillna("")
    .str.split(", ")
)

business_df[
    ["categories", "category_list"]
].head()

,categories,category_list
0,"Doctors, Traditional Chinese Medicine, Naturop...","[Doctors, Traditional Chinese Medicine, Naturo..."
1,"Shipping Centers, Local Services, Notaries, Ma...","[Shipping Centers, Local Services, Notaries, M..."
2,"Department Stores, Shopping, Fashion, Home & G...","[Department Stores, Shopping, Fashion, Home & ..."
3,"Restaurants, Food, Bubble Tea, Coffee & Tea, B...","[Restaurants, Food, Bubble Tea, Coffee & Tea, ..."
4,"Brewpubs, Breweries, Food","[Brewpubs, Breweries, Food]"


In [11]:
# 7. 핵심 음식점 범위 생성
business_df["is_core_restaurant"] = (
    business_df["category_list"]
    .apply(lambda categories: "Restaurants" in categories)
)

core_restaurant_count = (
    business_df["is_core_restaurant"].sum()
)

core_restaurant_rate = (
    core_restaurant_count
    / len(business_df)
    * 100
)

print("전체 업체 수:", len(business_df))
print("Restaurants 포함 업체:", core_restaurant_count)
print(f"음식점 비율: {core_restaurant_rate:.2f}%")

전체 업체 수: 150346
Restaurants 포함 업체: 52268
음식점 비율: 34.77%


In [12]:
# 8. 확장 음식점 범위 생성
expanded_food_categories = {
    "Restaurants",
    "Cafes",
    "Coffee & Tea",
    "Bakeries",
    "Desserts",
    "Food Trucks",
    "Bars"
}

In [13]:
business_df["is_expanded_food"] = (
    business_df["category_list"]
    .apply(
        lambda categories: bool(
            set(categories)
            & expanded_food_categories
        )
    )
)

expanded_food_count = (
    business_df["is_expanded_food"].sum()
)

expanded_food_rate = (
    expanded_food_count
    / len(business_df)
    * 100
)

print("확장 음식점 관련 업체:", expanded_food_count)
print(f"확장 음식점 비율: {expanded_food_rate:.2f}%")

확장 음식점 관련 업체: 59471
확장 음식점 비율: 39.56%


In [14]:
# 9. 범위별 업체 수 비교
scope_summary_df = pd.DataFrame(
    [
        {
            "scope": "전체 업체",
            "business_count": len(business_df),
            "business_rate_pct": 100.0
        },
        {
            "scope": "Restaurants 포함",
            "business_count": core_restaurant_count,
            "business_rate_pct": round(
                core_restaurant_rate,
                2
            )
        },
        {
            "scope": "확장 음식점",
            "business_count": expanded_food_count,
            "business_rate_pct": round(
                expanded_food_rate,
                2
            )
        }
    ]
)

scope_summary_df


,scope,business_count,business_rate_pct
0,전체 업체,150346,100.00
1,Restaurants 포함,52268,34.77
2,확장 음식점,59471,39.56


In [15]:
# 10. 음식점 세부 카테고리 확인
restaurant_category_counts = (
    business_df.loc[
        business_df["is_core_restaurant"],
        "category_list"
    ]
    .explode()
    .value_counts()
    .reset_index()
)

restaurant_category_counts.columns = [
    "category",
    "business_count"
]

restaurant_category_counts.head(30)

,category,business_count
0,Restaurants,52268
1,Food,15472
2,Nightlife,8723
3,Sandwiches,8366
4,Bars,8337
5,American (Traditional),8139
6,Pizza,7093
7,Fast Food,6472
8,Breakfast & Brunch,6239
9,American (New),6097


In [16]:
# 11. 지역별 음식점 수 확인
restaurant_by_state_df = (
    business_df[
        business_df["is_core_restaurant"]
    ]
    .groupby("state")
    .agg(
        business_count=("business_id", "nunique")
    )
    .sort_values(
        "business_count",
        ascending=False
    )
    .reset_index()
)

restaurant_by_state_df

,state,business_count
0,PA,12641
1,FL,8731
2,TN,4352
3,MO,4247
4,IN,4150
5,LA,3640
6,NJ,3341
7,AZ,2671
8,AB,2410
9,NV,1673


In [ ]:
# 도시별 상위 지역 확인
restaurant_by_city_df = (
    business_df[
        business_df["is_core_restaurant"]
    ]
    .groupby(
        ["state", "city"]
    )
    .agg(
        business_count=("business_id", "nunique")
    )
    .sort_values(
        "business_count",
        ascending=False
    )
    .reset_index()
)

restaurant_by_city_df.head(20)

,state,city,business_count
0,PA,Philadelphia,5852
1,FL,Tampa,2960
2,IN,Indianapolis,2862
3,TN,Nashville,2502
4,AZ,Tucson,2466
5,LA,New Orleans,2259
6,AB,Edmonton,2166
7,MO,Saint Louis,1790
8,NV,Reno,1286
9,ID,Boise,847


In [18]:
# 음식점 업체 데이터 확정 및 저장
# 1. 필요한 컬럼만 선택
restaurant_columns = [
    "business_id",
    "name",
    "address",
    "city",
    "state",
    "postal_code",
    "latitude",
    "longitude",
    "stars",
    "review_count",
    "is_open",
    "categories"
]

In [19]:
restaurant_business_df = (
    business_df.loc[
        business_df["is_core_restaurant"],
        restaurant_columns
    ]
    .copy()
    .reset_index(drop=True)
)

print(
    "음식점 업체 데이터 크기:",
    restaurant_business_df.shape
)

restaurant_business_df.head()

음식점 업체 데이터 크기: (52268, 12)


,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,categories
0,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"Restaurants, Food, Bubble Tea, Coffee & Tea, B..."
1,CF33F8-E6oudUQ46HnavjQ,Sonic Drive-In,615 S Main St,Ashland City,TN,37015,36.269593,-87.058943,2.0,6,1,"Burgers, Fast Food, Sandwiches, Food, Ice Crea..."
2,k0hlBqXX-Bt0vf1op7Jr1w,Tsevi's Pub And Grill,8025 Mackenzie Rd,Affton,MO,63123,38.565165,-90.321087,3.0,19,0,"Pubs, Restaurants, Italian, Bars, American (Tr..."
3,bBDDEgkFA1Otx9Lfe7BZUQ,Sonic Drive-In,2312 Dickerson Pike,Nashville,TN,37207,36.208102,-86.768170,1.5,10,1,"Ice Cream & Frozen Yogurt, Fast Food, Burgers,..."
4,eEOYSgkmpB90uNA7lDOMRA,Vietnamese Food Truck,,Tampa Bay,FL,33602,27.955269,-82.456320,4.0,10,1,"Vietnamese, Food, Restaurants, Food Trucks"


In [20]:
# 2. 좌표 결측 및 범위 검사
coordinate_validation_df = pd.DataFrame(
    [
        {
            "validation": "위도 결측",
            "count": restaurant_business_df[
                "latitude"
            ].isna().sum()
        },
        {
            "validation": "경도 결측",
            "count": restaurant_business_df[
                "longitude"
            ].isna().sum()
        },
        {
            "validation": "위도 범위 오류",
            "count": (
                ~restaurant_business_df[
                    "latitude"
                ].between(-90, 90)
            ).sum()
        },
        {
            "validation": "경도 범위 오류",
            "count": (
                ~restaurant_business_df[
                    "longitude"
                ].between(-180, 180)
            ).sum()
        }
    ]
)

coordinate_validation_df

,validation,count
0,위도 결측,0
1,경도 결측,0
2,위도 범위 오류,0
3,경도 범위 오류,0


In [21]:
# 3. 주요 컬럼 결측 검사
restaurant_missing_df = (
    restaurant_business_df[
        [
            "business_id",
            "city",
            "state",
            "postal_code",
            "latitude",
            "longitude",
            "categories"
        ]
    ]
    .isna()
    .sum()
    .reset_index()
)

restaurant_missing_df.columns = [
    "column",
    "missing_count"
]

restaurant_missing_df["missing_rate_pct"] = (
    restaurant_missing_df["missing_count"]
    / len(restaurant_business_df)
    * 100
).round(4)

restaurant_missing_df

,column,missing_count,missing_rate_pct
0,business_id,0,0.0
1,city,0,0.0
2,state,0,0.0
3,postal_code,0,0.0
4,latitude,0,0.0
5,longitude,0,0.0
6,categories,0,0.0


In [22]:
# 4. 음식점 데이터 저장
RESTAURANT_BUSINESS_PATH = (
    INTERIM_DIR
    / "restaurant_businesses.parquet"
)

restaurant_business_df.to_parquet(
    RESTAURANT_BUSINESS_PATH,
    index=False
)

print("저장 경로:", RESTAURANT_BUSINESS_PATH)
print(
    "저장 파일 크기:",
    round(
        RESTAURANT_BUSINESS_PATH.stat().st_size
        / (1024 ** 2),
        2
    ),
    "MB"
)

저장 경로: C:\Users\playdata2\SKN34-2nd-5Team\data\interim\restaurant_businesses.parquet
저장 파일 크기: 4.53 MB


In [23]:
# 5. 검증 결과표 저장
scope_summary_df.to_csv(
    REPORT_TABLE_DIR
    / "restaurant_scope_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

restaurant_category_counts.to_csv(
    REPORT_TABLE_DIR
    / "restaurant_category_counts.csv",
    index=False,
    encoding="utf-8-sig"
)

restaurant_by_state_df.to_csv(
    REPORT_TABLE_DIR
    / "restaurant_by_state.csv",
    index=False,
    encoding="utf-8-sig"
)

restaurant_by_city_df.to_csv(
    REPORT_TABLE_DIR
    / "restaurant_by_city.csv",
    index=False,
    encoding="utf-8-sig"
)

coordinate_validation_df.to_csv(
    REPORT_TABLE_DIR
    / "restaurant_coordinate_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

print("음식점 범위 검증 결과 저장 완료")

음식점 범위 검증 결과 저장 완료
